# Day 045 Project: Sales ETL Pipeline

## What You're Building

A complete ETL pipeline that:
1. Extracts raw sales data from a CSV string
2. Validates and filters out bad records
3. Transforms (cleans/normalizes) valid records
4. Loads them into a SQLAlchemy-backed database
5. Queries the database to produce a category and region summary

**Deliverable:** `run_pipeline` succeeds, `stats` shows correct counts, and `_run_project_checks()` passes all 5 checks.

## Project Requirements

1. Run `run_pipeline(CSV_SOURCE, session)` and store the result in `stats`
2. Print the pipeline stats
3. Query `Sale` rows by category using `select(Sale).where(...)`
4. Produce a category summary using `session.execute(text(sql))`
5. Produce a region summary

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import csv
import io
from sqlalchemy import create_engine, String, Float, select, text
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session
from sqlalchemy.pool import StaticPool


class Base(DeclarativeBase):
    pass


class Sale(Base):
    __tablename__ = 'sales'
    id:       Mapped[int]   = mapped_column(primary_key=True)
    date:     Mapped[str]   = mapped_column(String(20))
    product:  Mapped[str]   = mapped_column(String(100))
    category: Mapped[str]   = mapped_column(String(50))
    amount:   Mapped[float] = mapped_column()
    region:   Mapped[str]   = mapped_column(String(50))

    def __repr__(self):
        return f'Sale(id={self.id}, product={self.product!r}, amount={self.amount})'


def setup_engine(url='sqlite:///:memory:'):
    engine = create_engine(
        url,
        connect_args={'check_same_thread': False},
        poolclass=StaticPool,
    )
    Base.metadata.create_all(engine)
    return engine


def extract(csv_text: str) -> list:
    reader = csv.DictReader(io.StringIO(csv_text))
    return list(reader)


def validate_record(record: dict) -> bool:
    required = ['date', 'product', 'amount']
    for field in required:
        if not record.get(field, '').strip():
            return False
    try:
        float(record['amount'])
    except (ValueError, TypeError):
        return False
    return True


def transform_record(record: dict) -> dict:
    return {
        'date':     record['date'].strip(),
        'product':  record['product'].strip(),
        'category': record.get('category', '').strip(),
        'amount':   round(float(record['amount']), 2),
        'region':   record.get('region', '').strip().title(),
    }


def load(session, records: list) -> int:
    sales = [Sale(**r) for r in records]
    session.add_all(sales)
    session.commit()
    return len(sales)


def run_pipeline(csv_text: str, session) -> dict:
    raw_records  = extract(csv_text)
    valid        = [r for r in raw_records if validate_record(r)]
    transformed  = [transform_record(r) for r in valid]
    loaded_count = load(session, transformed)
    return {
        'extracted': len(raw_records),
        'loaded':    loaded_count,
        'skipped':   len(raw_records) - loaded_count,
    }


CSV_SOURCE = (
    'date,product,category,amount,region\n'
    '2024-02-01,Laptop Pro,Electronics,1299.99,east\n'
    '2024-02-02,Wireless Mouse,Electronics,39.99,west\n'
    '2024-02-03,Standing Desk,Furniture,549.00,east\n'
    '2024-02-04,Bookshelf,Furniture,229.00,north\n'
    '2024-02-05,Gel Pens 12pk,Stationery,14.99,south\n'
    '2024-02-06,Monitor 27in,Electronics,449.99,west\n'
    '2024-02-07,,Electronics,89.99,east\n'
    '2024-02-08,USB Hub,Electronics,twenty,south\n'
    '2024-02-09,Ergonomic Chair,Furniture,699.00,north\n'
    '2024-02-10,Mechanical Keyboard,Electronics,129.99,east\n'
    '2024-02-11,Sticky Notes,Stationery,6.99,west\n'
    '2024-02-12,Webcam HD,Electronics,199.99,south\n'
    '2024-02-13,Desk Lamp,Furniture,59.99,north\n'
    '2024-02-14,Stapler,Stationery,,\n'
    '2024-02-15,Notebook Set,Stationery,22.99,west\n'
)

engine  = setup_engine()
session = Session(engine)
print('ETL pipeline ready.')

## Step 1 — Run the Pipeline

In [ ]:
stats = run_pipeline(CSV_SOURCE, session)
print(f'Extracted: {stats["extracted"]}')
print(f'Loaded:    {stats["loaded"]}')
print(f'Skipped:   {stats["skipped"]}')

## Step 2 — Browse by Category

In [ ]:
# Query a specific category using ORM select
# electronics = session.execute(
#     select(Sale).where(Sale.category == 'Electronics')
# ).scalars().all()
# print(f'Electronics ({len(electronics)} items):')
# for s in electronics:
#     print(f'  {s}')

## Step 3 — Category Summary (SQL)

In [ ]:
# Use session.execute(text(sql)) to get a GROUP BY summary
# sql = (
#     'SELECT category, COUNT(*) as count, '
#     'ROUND(SUM(amount), 2) as total '
#     'FROM sales GROUP BY category ORDER BY total DESC'
# )
# rows = session.execute(text(sql)).mappings().all()
# print('Category summary:')
# for row in rows:
#     print(f'  {row["category"]}: {row["count"]} sales, ${row["total"]:.2f}')

## Step 4 — Region Summary

In [ ]:
# TODO: produce a region summary similar to Step 3
# SELECT region, COUNT(*) as count, ROUND(SUM(amount), 2) as total
# FROM sales GROUP BY region ORDER BY total DESC

## Project Checks

In [ ]:
def _run_project_checks():
    total = 5
    passed = 0

    # Check 1: stats dict exists with correct keys
    try:
        assert 'stats' in globals()
        assert {'extracted', 'loaded', 'skipped'} <= set(stats.keys())
        passed += 1; print(f'\u2705 Check 1: stats = {stats}')
    except Exception as e:
        print(f'\u274c Check 1: {e}')

    # Check 2: extracted == 15 (total CSV rows)
    try:
        assert stats['extracted'] == 15, \
            f'expected 15 extracted, got {stats["extracted"]}'
        passed += 1; print('\u2705 Check 2: extracted=15')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: 3 invalid rows skipped, 12 loaded
    try:
        assert stats['skipped'] == 3, \
            f'expected skipped=3, got {stats["skipped"]}'
        assert stats['loaded'] == 12, \
            f'expected loaded=12, got {stats["loaded"]}'
        passed += 1; print('\u2705 Check 3: skipped=3, loaded=12')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: 12 Sale rows in DB
    try:
        db_count = len(session.execute(select(Sale)).scalars().all())
        assert db_count == 12, \
            f'expected 12 rows in DB, got {db_count}'
        passed += 1; print(f'\u2705 Check 4: {db_count} Sale rows in DB')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: region values are title-cased (not lowercase)
    try:
        regions = [
            s.region for s in
            session.execute(select(Sale)).scalars().all()
        ]
        assert all(r == r.title() for r in regions if r), \
            f'some regions not title-cased: {[r for r in regions if r != r.title()]}'
        passed += 1; print(f'\u2705 Check 5: all regions title-cased ({set(regions)})')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Project complete!')
    print(f'\nScore: {passed}/{total}')


_run_project_checks()

## Bonus Challenges

- Add a `deduplicate(records)` step that drops records with the same `(date, product)` pair before loading
- Make the pipeline idempotent: add `INSERT OR IGNORE` logic so that running the pipeline twice doesn't double-load records
- Add a `report(session)` function that uses `pd.read_sql_query` with `engine.connect()` to return a DataFrame summary
- On Day 46 you will apply time-series analysis to the `date` column — the ETL pipeline you built today is the data source